In [18]:
from shapely import wkt
from shapely.wkt import dumps
from shapely.geometry import Point
from shapely.geometry.base import BaseGeometry
from tqdm import tqdm
import geopandas as gpd
import pandas as pd
import numpy as np
from dotenv import load_dotenv
import os
from langchain_community.graphs import Neo4jGraph
import warnings
import glob

# Data and config

In [28]:
# shizuoka data loading
# ward = gpd.read_file('/Users/liuziheng/Documents/Shizuoka UrbanKG/Processed data/Administrative/Shizuoka.shp')
# area = gpd.read_file('/Users/liuziheng/Documents/Shizuoka UrbanKG/Processed data/Administrative/Shizuoka_towns.shp')
# poi = pd.read_csv('/Users/liuziheng/Documents/Shizuoka UrbanKG/Processed data/POI/Shizuoka_POI.csv')
# road = gpd.read_file('/Users/liuziheng/Documents/Shizuoka UrbanKG/Processed data/Railway/DRM_Shizuoka.shp')
# block_data_path = '/Users/liuziheng/Documents/Raw data/Block center point data/Shizuoka'

/var/folders/l1/w_3r8_td2fs1mtp6m61wrpw80000gn/T/ipykernel_63150/4253611300.py:4: DtypeWarning: Columns (13) have mixed types. Specify dtype option on import or set low_memory=False.
  poi = pd.read_csv('/Users/liuziheng/Documents/Shizuoka UrbanKG/Processed data/POI/Shizuoka_POI.csv')


In [ ]:
# susono data loading
# ward = gpd.read_file('/Users/liuziheng/Documents/Susono UrbanKG/Processed data/Administrative/Susuno.shp')
# area = gpd.read_file('/Users/liuziheng/Documents/Susono UrbanKG/Processed data/Administrative/Susono_towns.shp')
# poi = pd.read_csv('/Users/liuziheng/Documents/Susono UrbanKG/Processed data/POI/Susono_POI.csv')
# road = gpd.read_file('/Users/liuziheng/Documents/Susono UrbanKG/Processed data/Railway/DRM_Susono.shp')
# block_data_path = '/Users/liuziheng/Documents/Raw data/Block center point data/Susono'

In [19]:
# tokyo data loading
# ward = gpd.read_file('/Users/liuziheng/Documents/Tokyo UrbanKG/Processed data/Administrative/Tokyo23.shp')
# area = gpd.read_file('/Users/liuziheng/Documents/Tokyo UrbanKG/Processed data/Administrative/Tokyo23_towns.shp')
# poi = pd.read_csv('/Users/liuziheng/Documents/Tokyo UrbanKG/Processed data/POI/Tokyo_POI.csv')
# road = gpd.read_file('/Users/liuziheng/Documents/Tokyo UrbanKG/Processed data/Railway/DRM_Tokyo.shp')
# block_data_path = '/Users/liuziheng/Documents/Raw data/Block center point data/Tokyo'

/var/folders/l1/w_3r8_td2fs1mtp6m61wrpw80000gn/T/ipykernel_63150/1954938908.py:4: DtypeWarning: Columns (13) have mixed types. Specify dtype option on import or set low_memory=False.
  poi = pd.read_csv('/Users/liuziheng/Documents/Tokyo UrbanKG/Processed data/POI/Tokyo_POI.csv')


In [20]:
# Load from environment
load_dotenv('.env', override=True)
NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")

def get_kg(database: str):
    return Neo4jGraph(
        url=NEO4J_URI,
        username=NEO4J_USERNAME,
        password=NEO4J_PASSWORD,
        database=database
    )

In [41]:
kg = get_kg("tokyo")

# Entity construction under normalization

To unify the address formats for Blocks and POIs, the following normalization steps are required:

- Block: Convert Kanji numerals in Chome to Arabic numerals (e.g., 永田町一丁目 → 永田町 1丁目).

- POI: Convert full-width spaces to half-width spaces.

- Unified Format: Tokyo-to [Ward] [Town] [Chome] [Block Number]

In [24]:
import re

def zen_to_han(text):
    """
    Convert full-width digits and common symbols to half-width
    """
    if pd.isna(text) or not text:
        return text
    
    # Mapping
    zen_digits = '０１２３４５６７８９'
    han_digits = '0123456789'
    
    zen_symbols = '−－ー‐'  
    han_symbols = '----'
    
    text = str(text)
    
    for zen, han in zip(zen_digits, han_digits):
        text = text.replace(zen, han)
    
    for zen in zen_symbols:
        text = text.replace(zen, '-')
    
    return text


def parse_address(address):
    """
    Analysis of standarized Japanese addresses
    Return：{pref, city, town, chome, block, banchi}
    """
    if pd.isna(address) or not address:
        return None
    
    # Standard format: 東京都 千代田区 永田町 1丁目 6-2
    parts = str(address).strip().split()
    
    result = {
        'pref': None,
        'city': None, 
        'town': None,
        'chome': None,
        'block': None,
        'banchi': None
    }
    
    if len(parts) < 3:
        return None
    
    result['pref'] = parts[0]  
    result['city'] = parts[1]  
    
    # left parts
    remaining = ' '.join(parts[2:])
    
    # Matching format such as "永田町 1丁目 6-2" or "永田町 1丁目 6"
    pattern = r'(.+?)\s+(\d+)丁目\s+([\d\-]+)'
    match = re.search(pattern, remaining)
    
    if match:
        result['town'] = match.group(1)  
        result['chome'] = match.group(2) + '丁目'  
        block_part = match.group(3)  
        
        if '-' in block_part:
            block_banchi = block_part.split('-')
            result['block'] = block_banchi[0]  # 6
            result['banchi'] = '-'.join(block_banchi[1:])  # 2
        else:
            result['block'] = block_part  # 6
    else:
        # Without chome 丁目
        pattern2 = r'(.+?)\s+([\d\-]+)'
        match2 = re.search(pattern2, remaining)
        if match2:
            result['town'] = match2.group(1)
            block_part = match2.group(2)
            if '-' in block_part:
                block_banchi = block_part.split('-')
                result['block'] = block_banchi[0]
                result['banchi'] = '-'.join(block_banchi[1:])
            else:
                result['block'] = block_part
    
    return result


def normalize_address_block(pref, city, choume, block_num):

    kanji_to_num = {
        '一': '1', '二': '2', '三': '3', '四': '4', '五': '5',
        '六': '6', '七': '7', '八': '8', '九': '9', '十': '10'
    }
    
    match = re.search(r'(.+?)([一二三四五六七八九十]+)丁目', str(choume))
    if match:
        base_name = match.group(1)
        kanji_num = match.group(2)
        
        if '十' in kanji_num:
            if kanji_num == '十':
                arabic_num = '10'
            elif kanji_num.startswith('十'):
                arabic_num = str(10 + int(kanji_to_num.get(kanji_num[1], 0)))
            else:
                arabic_num = str(int(kanji_to_num.get(kanji_num[0], 0)) * 10)
                if len(kanji_num) > 1 and kanji_num[1] != '十':
                    arabic_num = str(int(arabic_num) + int(kanji_to_num.get(kanji_num[-1], 0)))
        else:
            arabic_num = kanji_to_num.get(kanji_num, kanji_num)
        
        standardized_choume = f"{base_name} {arabic_num}丁目"
    else:
        standardized_choume = str(choume)
    
    block_num = zen_to_han(str(block_num))
    
    if '-' in block_num:
        block_num = block_num.split('-')[0]
    
    address = f"{pref} {city} {standardized_choume} {block_num}"
    return address

def normalize_address_poi(address_str):
    if pd.isna(address_str):
        return ""
    
    address = str(address_str)
    address = address.replace('　', ' ')
    address = zen_to_han(address)
    address = re.sub(r'\s+', ' ', address)
    
    return address.strip()

def standardize_area_town_name(town_name):

    if pd.isna(town_name) or not town_name:
        return town_name
    
    kanji_to_num = {
        '一': '1', '二': '2', '三': '3', '四': '4', '五': '5',
        '六': '6', '七': '7', '八': '8', '九': '9', '十': '10'
    }
    
    match = re.search(r'(.+?)([一二三四五六七八九十]+)丁目', str(town_name))
    if match:
        base_name = match.group(1)
        kanji_num = match.group(2)
        
        if '十' in kanji_num:
            if kanji_num == '十':
                arabic_num = '10'
            elif kanji_num.startswith('十'):
                arabic_num = str(10 + int(kanji_to_num.get(kanji_num[1], 0)))
            else:
                arabic_num = str(int(kanji_to_num.get(kanji_num[0], 0)) * 10)
                if len(kanji_num) > 1 and kanji_num[1] != '十':
                    arabic_num = str(int(arabic_num) + int(kanji_to_num.get(kanji_num[-1], 0)))
        else:
            arabic_num = kanji_to_num.get(kanji_num, kanji_num)
        return f"{base_name} {arabic_num}丁目"
    else:
        return str(town_name)

def enrich_blocks_with_geometry(kg, block_data_path, batch_size=5000):

    csv_files = glob.glob(os.path.join(block_data_path, '*/*.csv'))
    
    geometry_updates = []
    matched_count = 0
    unmatched_count = 0
    
    for csv_file in sorted(csv_files):
        try:
            df = pd.read_csv(csv_file, encoding='shift-jis')
            ward_name = df['市区町村名'].iloc[0] if len(df) > 0 else "N/A"
            
            for idx, row in tqdm(df.iterrows(), total=len(df)):
                block_address = normalize_address_block(
                    row['都道府県名'],
                    row['市区町村名'],
                    row['大字_丁目名'],
                    row['街区符号_地番']
                )
                
                point = Point(row['経度'], row['緯度'])
                geometry_wkt = dumps(point)
                
                geometry_updates.append({
                    'address': block_address,
                    'geometry': geometry_wkt
                })
                matched_count += 1
                
        except Exception as e:
            print(f"Error processing {csv_file}: {e}")
            continue
    
    query = """
    UNWIND $batch AS row
    MATCH (b:Block {address: row.address})
    SET b.geometry = row.geometry
    """
    
    updated_count = 0
    for i in range(0, len(geometry_updates), batch_size):
        batch = geometry_updates[i:i+batch_size]
        result = kg.query(query, params={'batch': batch})
        updated_count += len(batch)




def create_ward(kg, prefecture, ward_name, ward_code, geometry):
    '''
    Entity 1: Ward
    '''
    geometry_wkt = dumps(geometry)
    
    query = (
        "MERGE (w:Ward {ward_code: $ward_code}) "
        "SET w.name = $ward_name, "
        "    w.prefecture = $prefecture, "
        "    w.geometry = $geometry"
    )
    
    kg.query(query, params={
        "ward_code": str(ward_code),
        "ward_name": ward_name,
        "prefecture": prefecture,
        "geometry": geometry_wkt
    })
    
def create_area(kg, prefecture, ward_name, town_name, town_code, geometry):
    '''
    Entity 2: Area
    '''
    geometry_wkt = dumps(geometry)
    standardized_town_name = standardize_area_town_name(town_name)

    query = (
        "MERGE (a:Area {town_code: $town_code}) "
        "SET a.name = $town_name, "
        "    a.prefecture = $prefecture, "
        "    a.ward_name = $ward_name, "
        "    a.geometry = $geometry"
    )
    
    kg.query(query, params={
        "town_code": str(town_code),
        "town_name": standardized_town_name,
        "ward_name": ward_name,
        "prefecture": prefecture,
        "geometry": geometry_wkt
    })

def create_pois_batch(kg, poi_data, batch_size=1000):
    '''
    Entity 3: POI
    '''

    query = """
    UNWIND $batch AS row
    MERGE (p:POI {poi_id: row.poi_id})
    SET p.name = row.name,
        p.address = row.address,
        p.geometry = row.geometry,
        p.ptypes = row.ptypes
    FOREACH (_ IN CASE WHEN row.is_school THEN [1] ELSE [] END |
        SET p:School
    )
    """
    
    batch = []
    total = len(poi_data)
    
    for idx, row in poi_data.iterrows():
        ptypes = row['PTYPE'].split(',') if isinstance(row['PTYPE'], str) else (row['PTYPE'] or [])
        normalized_address = normalize_address_poi(row['ADD'])
        
        batch.append({
            "poi_id": row['poi_id'],
            "name": row['DN'],
            "address": normalized_address,
            "geometry": row['geometry'],
            "ptypes": ptypes,
            "is_school": False  
        })
        
        if len(batch) >= batch_size:
            kg.query(query, params={"batch": batch})
            print(f"  POI construction {idx + 1}/{total}")
            batch = []
    
    if batch:
        kg.query(query, params={"batch": batch})
        print(f"  POI construction {total}/{total}")


def create_roads_batch(kg, road_data, batch_size=1000):
    '''
    Entity 4: Road
    '''
    query = """
    UNWIND $batch AS row
    MERGE (r:Road {road_id: row.road_id})
    SET r.rtype = row.rtype,
        r.length = row.length,
        r.geometry = row.geometry
    """
    
    batch = []
    total = len(road_data)
    
    for idx, row in road_data.iterrows():
        batch.append({
            "road_id": str(row['gid']),
            "rtype": str(row['rdclasscd']),
            "length": row['length'],
            "geometry": dumps(row['geometry']) 
        })
        
        if len(batch) >= batch_size:
            kg.query(query, params={"batch": batch})
            print(f"  Road construction {idx + 1}/{total}")
            batch = []
    
    if batch:
        kg.query(query, params={"batch": batch})
        print(f"  Road construction {total}/{total}")


def create_blocks_from_poi(kg, poi_data, batch_size=5000):
    '''
    Entity 5(Dummy): Block
    '''
    
    block_addresses = {}  # {block_address: count}
    
    for idx, row in tqdm(poi_data.iterrows(), total=len(poi_data)):
        poi_address = normalize_address_poi(row['ADD'])
        parsed = parse_address(poi_address)
        
        if not parsed or not parsed['block']:
            continue
        
        block_parts = [
            parsed['pref'],
            parsed['city'],
            parsed['town'],
            parsed['chome'] if parsed['chome'] else '',
            parsed['block']
        ]
        
        block_address = ' '.join([p for p in block_parts if p])
        
        if block_address not in block_addresses:
            block_addresses[block_address] = 0
        block_addresses[block_address] += 1
    
    query = """
    UNWIND $batch AS row
    MERGE (b:Block {address: row.address})
    ON CREATE SET b.poi_count = row.poi_count
    ON MATCH SET b.poi_count = row.poi_count
    """
    
    batch = []
    for block_address, count in block_addresses.items():
        batch.append({
            'address': block_address,
            'poi_count': count
        })
        
        if len(batch) >= batch_size:
            kg.query(query, params={'batch': batch})
            batch = []
    
    if batch:
        kg.query(query, params={'batch': batch})
    
    return len(block_addresses)


def create_constraints(kg):
    constraints = [

        """
        CREATE CONSTRAINT ward_code_unique IF NOT EXISTS
        FOR (w:Ward) REQUIRE w.ward_code IS UNIQUE
        """,
        
        """
        CREATE CONSTRAINT town_code_unique IF NOT EXISTS
        FOR (a:Area) REQUIRE a.town_code IS UNIQUE
        """,
        
        """
        CREATE CONSTRAINT poi_id_unique IF NOT EXISTS
        FOR (p:POI) REQUIRE p.poi_id IS UNIQUE
        """,
        
        """
        CREATE CONSTRAINT road_id_unique IF NOT EXISTS
        FOR (r:Road) REQUIRE r.road_id IS UNIQUE
        """,
        
        """
        CREATE CONSTRAINT block_address_unique IF NOT EXISTS
        FOR (b:Block) REQUIRE b.address IS UNIQUE
        """,

        """
        CREATE CONSTRAINT cate_name_unique IF NOT EXISTS
        FOR (c:Cate) REQUIRE c.name IS UNIQUE
        """
    ]
    
    for constraint_query in constraints:
        try:
            kg.query(constraint_query)
            print(f"Constraint creation successful")
        except Exception as e:
            print(f"Constraint creation failed or already exists: {e}")
    
    result = kg.query("SHOW CONSTRAINTS")
    print(f"\nCurrently constraints: {len(result)}")
    for constraint in result:
        print(f"  - {constraint}")


def _ensure_gdf_crs(gdf, default_crs="EPSG:4326"):
    if gdf.crs is None:
        return gdf.set_crs(default_crs)
    return gdf


def create_relationships_PLA_by_spatial(kg, poi_dataframe, area_dataframe, batch_size=5000):
    """
    Relation: (POI)-[:locatesAt]->(Area) by spatial containment.
    """
    from shapely import wkt as _wkt

    poi_rows = []
    for _, row in poi_dataframe.iterrows():
        try:
            geom = _wkt.loads(row['geometry']) if isinstance(row['geometry'], str) else row['geometry']
            poi_rows.append({'poi_id': int(row['poi_id']), 'geometry': geom})
        except Exception:
            continue

    if not poi_rows:
        print("No POI geometry available for PLA relationship")
        return

    poi_gdf = gpd.GeoDataFrame(poi_rows, geometry='geometry', crs="EPSG:4326")
    area_gdf = _ensure_gdf_crs(area_dataframe).to_crs("EPSG:4326")

    join_df = gpd.sjoin(
        poi_gdf[['poi_id', 'geometry']],
        area_gdf[['KEY_CODE', 'geometry']],
        how='inner',
        predicate='within'
    )[['poi_id', 'KEY_CODE']].drop_duplicates()

    rows = [
        {'poi_id': int(r.poi_id), 'area_code': str(r.KEY_CODE)}
        for r in join_df.itertuples(index=False)
    ]

    query = """
    UNWIND $batch AS row
    MATCH (p:POI {poi_id: row.poi_id})
    MATCH (a:Area {town_code: row.area_code})
    MERGE (p)-[:locatesAt]->(a)
    """

    for i in range(0, len(rows), batch_size):
        kg.query(query, params={'batch': rows[i:i+batch_size]})
    print(f"PLA created: {len(rows)}")


def create_relationships_ABW_by_spatial(kg, area_dataframe, ward_dataframe, batch_size=5000):
    """
    Relation: (Area)-[:belongsTo]->(Ward) by spatial containment.
    """
    area_gdf = _ensure_gdf_crs(area_dataframe).to_crs("EPSG:4326")
    ward_gdf = _ensure_gdf_crs(ward_dataframe).to_crs("EPSG:4326")

    join_df = gpd.sjoin(
        area_gdf[['KEY_CODE', 'geometry']],
        ward_gdf[['N03_007', 'geometry']],
        how='inner',
        predicate='within'
    )[['KEY_CODE', 'N03_007']].drop_duplicates()

    rows = [
        {'area_code': str(r.KEY_CODE), 'ward_code': str(r.N03_007)}
        for r in join_df.itertuples(index=False)
    ]

    query = """
    UNWIND $batch AS row
    MATCH (a:Area {town_code: row.area_code})
    MATCH (w:Ward {ward_code: row.ward_code})
    MERGE (a)-[:belongsTo]->(w)
    """

    for i in range(0, len(rows), batch_size):
        kg.query(query, params={'batch': rows[i:i+batch_size]})
    print(f"ABW created: {len(rows)}")


def create_relationships_RIA_by_spatial(kg, road_gdf, area_gdf, predicate='intersects', batch_size=5000):
    """
    Relation: (Road)-[:in]->(Area) by spatial join.
    """
    road_gdf = _ensure_gdf_crs(road_gdf).to_crs("EPSG:4326")
    area_gdf = _ensure_gdf_crs(area_gdf).to_crs("EPSG:4326")

    pairs = gpd.sjoin(
        road_gdf[['gid', 'geometry']],
        area_gdf[['KEY_CODE', 'geometry']],
        how='inner',
        predicate=predicate
    )[['gid', 'KEY_CODE']].drop_duplicates()

    rows = [{'road_id': str(r.gid), 'area_code': str(r.KEY_CODE)} for r in pairs.itertuples(index=False)]

    query = """
    UNWIND $batch AS row
    MATCH (r:Road {road_id: row.road_id})
    MATCH (a:Area {town_code: row.area_code})
    MERGE (r)-[:in]->(a)
    """

    for i in range(0, len(rows), batch_size):
        kg.query(query, params={'batch': rows[i:i+batch_size]})
    print(f"RIA created: {len(rows)}")


def create_relationships_ABA_by_spatial(kg, area_dataframe, batch_size=5000):
    """
    Relation: (Area)-[:borderBy]->(Area) by touches.
    """
    from shapely.strtree import STRtree

    area_gdf = _ensure_gdf_crs(area_dataframe).to_crs("EPSG:4326").reset_index(drop=True)
    geoms = list(area_gdf['geometry'])
    tree = STRtree(geoms)

    rels = []
    for i, area in area_gdf.iterrows():
        g = area['geometry']
        c1 = str(area['KEY_CODE'])
        for j in tree.query(g):
            if i == j:
                continue
            g2 = geoms[j]
            c2 = str(area_gdf.iloc[j]['KEY_CODE'])
            if g.touches(g2):
                if c1 < c2:
                    rels.append({'a1': c1, 'a2': c2})

    query = """
    UNWIND $batch AS row
    MATCH (a1:Area {town_code: row.a1})
    MATCH (a2:Area {town_code: row.a2})
    MERGE (a1)-[:borderBy]->(a2)
    MERGE (a2)-[:borderBy]->(a1)
    """

    for i in range(0, len(rels), batch_size):
        kg.query(query, params={'batch': rels[i:i+batch_size]})
    print(f"ABA created pairs: {len(rels)}")


def create_relationships_PLB_by_address(kg, poi_dataframe, batch_size=5000):
    """
    Relation: (POI)-[:locatesAt]->(Block) by normalized address parsing.
    """
    rows = []
    for _, row in tqdm(poi_dataframe.iterrows(), total=len(poi_dataframe)):
        addr = normalize_address_poi(row.get('address'))
        parsed = parse_address(addr)
        if not parsed or not parsed.get('block'):
            continue

        block_parts = [
            parsed['pref'],
            parsed['city'],
            parsed['town'],
            parsed['chome'] if parsed.get('chome') else '',
            parsed['block']
        ]
        block_address = ' '.join([p for p in block_parts if p])

        rows.append({'poi_id': int(row['poi_id']), 'block_address': block_address})

    query = """
    UNWIND $batch AS row
    MATCH (p:POI {poi_id: row.poi_id})
    MATCH (b:Block {address: row.block_address})
    MERGE (p)-[:locatesAt]->(b)
    """

    for i in range(0, len(rows), batch_size):
        kg.query(query, params={'batch': rows[i:i+batch_size]})
    print(f"PLB created: {len(rows)}")


def create_relationships_BTA_by_address(kg, area_dataframe, batch_size=5000):
    """
    Relation: (Block)-[:belongsTo]->(Area) by normalized address key matching.
    """
    area_mapping = {}
    for _, row in area_dataframe.iterrows():
        std_name = standardize_area_town_name(str(row['S_NAME']))
        area_key = f"{row['PREF_NAME']} {row['CITY_NAME']} {std_name}".strip()
        area_mapping[area_key] = str(row['KEY_CODE'])

    blocks = kg.query("MATCH (b:Block) RETURN b.address as address")

    rels = []
    for r in tqdm(blocks, total=len(blocks)):
        baddr = r['address']
        parsed = parse_address(baddr)
        if not parsed:
            continue

        town_std = standardize_area_town_name(parsed['town']) if parsed.get('town') else None
        if not town_std:
            continue

        if parsed.get('chome'):
            area_key = f"{parsed['pref']} {parsed['city']} {town_std}{parsed['chome']}"
        else:
            area_key = f"{parsed['pref']} {parsed['city']} {town_std}"

        area_code = area_mapping.get(area_key)
        if area_code:
            rels.append({'block_address': baddr, 'area_code': area_code})

    query = """
    UNWIND $batch AS row
    MATCH (b:Block {address: row.block_address})
    MATCH (a:Area {town_code: row.area_code})
    MERGE (b)-[:belongsTo]->(a)
    """

    for i in range(0, len(rels), batch_size):
        kg.query(query, params={'batch': rels[i:i+batch_size]})
    print(f"BTA created: {len(rels)}")


def create_relationships_PNR_by_distance(kg, poi_dataframe, road_gdf, max_distance=20, batch_size=5000):
    """
    Relation: (POI)-[:nearby]->(Road) by nearest distance threshold (meters).
    """
    from shapely import wkt as _wkt

    poi_rows = []
    for _, row in poi_dataframe.iterrows():
        try:
            geom = _wkt.loads(row['geometry']) if isinstance(row['geometry'], str) else row['geometry']
            poi_rows.append({'poi_id': int(row['poi_id']), 'geometry': geom})
        except Exception:
            continue

    if not poi_rows:
        print("No POI geometry available for PNR relationship")
        return

    poi_gdf = gpd.GeoDataFrame(poi_rows, geometry='geometry', crs="EPSG:4326")
    road_gdf = _ensure_gdf_crs(road_gdf).to_crs("EPSG:4326")

    proj_crs = poi_gdf.estimate_utm_crs()
    poi_proj = poi_gdf.to_crs(proj_crs)
    road_proj = road_gdf.to_crs(proj_crs)

    near_df = gpd.sjoin_nearest(
        poi_proj[['poi_id', 'geometry']],
        road_proj[['gid', 'geometry']],
        how='left',
        max_distance=max_distance,
        distance_col='dist_m'
    )[['poi_id', 'gid']].dropna().drop_duplicates()

    rows = [{'poi_id': int(r.poi_id), 'road_id': str(r.gid)} for r in near_df.itertuples(index=False)]

    query = """
    UNWIND $batch AS row
    MATCH (p:POI {poi_id: row.poi_id})
    MATCH (r:Road {road_id: row.road_id})
    MERGE (p)-[:nearby]->(r)
    """

    for i in range(0, len(rows), batch_size):
        kg.query(query, params={'batch': rows[i:i+batch_size]})
    print(f"PNR created: {len(rows)}")

categories_ranges = {
    'Factory': [(100000, 1900000)],
    'Business': [(1900000, 2200000), (2500000, 2700000), (2800000, 2900000)],
    'Transportation': [(2200000, 2200400)],

    'Amusement': [(2900000, 2919000), (3100000, 3122008), (3178000, 3212000)],
    'Meal': [(3123000, 3178000)],
    'Accommodation': [(3213000, 3226000)],

    'Medical_service': [(3300000, 3340000)],
    'School': [(3340000, 3340000), (3700000, 3800000)],
    'Welfare': [(3341000, 3400000)],
    'Life_related': [(3000000, 3008000), (3400000, 3600000), (3602000, 3615000)],
    'Car_service': [(3600000, 3602000), (3615000, 3615000)],
    'Government': [(3800000, 3900000)],
    'Shopping': [(3400000, 3540000), (3540012, 3561003)],

    'others': []
}


def _extract_busc_codes(busc_value):
    """Parse BUSC field into integer code list."""
    if busc_value is None:
        return []

    if isinstance(busc_value, str):
        raw_values = busc_value.split(',')
    elif isinstance(busc_value, (list, tuple, set, np.ndarray, pd.Series)):
        raw_values = list(busc_value)
    else:
        raw_values = [busc_value]

    codes = []
    for v in raw_values:
        if v is None:
            continue
        if isinstance(v, float) and pd.isna(v):
            continue

        token = str(v).strip()
        if not token:
            continue

        match = re.search(r'\d+', token)
        if not match:
            continue

        try:
            codes.append(int(match.group()))
        except Exception:
            continue

    return codes



def map_busc_to_cates(busc_codes, mapping=categories_ranges):
    """Map BUSC code(s) to Cate names based on range rules."""
    matched = set()

    for code in busc_codes:
        for cate, ranges in mapping.items():
            if cate == 'others':
                continue
            for low, high in ranges:
                if low <= code <= high:
                    matched.add(cate)
                    break

    if not matched:
        matched.add('others')

    return sorted(matched)



def create_relationships_PCT_by_ptype(kg, poi_dataframe, batch_size=5000):
    """
    Relation: (POI)-[:hasType]->(Cate) by BUSC range mapping.
    """
    rows = []

    for _, row in tqdm(poi_dataframe.iterrows(), total=len(poi_dataframe)):
        busc_value = row.get('BUSC') if 'BUSC' in row else row.get('busc')
        busc_codes = _extract_busc_codes(busc_value)
        cates = map_busc_to_cates(busc_codes)

        for cate in cates:
            rows.append({
                'poi_id': int(row['poi_id']),
                'cate': cate
            })

    query = """
    UNWIND $batch AS row
    MATCH (p:POI {poi_id: row.poi_id})
    MERGE (c:Cate {name: row.cate})
    MERGE (p)-[:hasType]->(c)
    """

    for i in range(0, len(rows), batch_size):
        kg.query(query, params={'batch': rows[i:i+batch_size]})

    print(f"PCT created: {len(rows)}")



In [34]:
create_constraints(kg)

Constraint creation successful
Constraint creation successful
Constraint creation successful
Constraint creation successful
Constraint creation successful
Constraint creation successful

Currently constraints: 7
  - {'id': 13, 'name': 'block_address_unique', 'type': 'UNIQUENESS', 'entityType': 'NODE', 'labelsOrTypes': ['Block'], 'properties': ['address'], 'ownedIndex': 'block_address_unique', 'propertyType': None}
  - {'id': 11, 'name': 'block_id_unique', 'type': 'UNIQUENESS', 'entityType': 'NODE', 'labelsOrTypes': ['Block'], 'properties': ['block_id'], 'ownedIndex': 'block_id_unique', 'propertyType': None}
  - {'id': 15, 'name': 'cate_name_unique', 'type': 'UNIQUENESS', 'entityType': 'NODE', 'labelsOrTypes': ['Cate'], 'properties': ['name'], 'ownedIndex': 'cate_name_unique', 'propertyType': None}
  - {'id': 7, 'name': 'poi_id_unique', 'type': 'UNIQUENESS', 'entityType': 'NODE', 'labelsOrTypes': ['POI'], 'properties': ['poi_id'], 'ownedIndex': 'poi_id_unique', 'propertyType': None}
  -

In [ ]:
for _, row in ward.iterrows():
    create_ward(
        kg=kg,
        prefecture=row['N03_001'],
        ward_name=row['N03_004'],
        ward_code=row['N03_007'],
        geometry=row['geometry']
    )

for _, row in area.iterrows():
    create_area(
        kg=kg,
        prefecture=row['PREF_NAME'],
        ward_name=row['CITY_NAME'],
        town_name=row['S_NAME'],
        town_code=row['KEY_CODE'],
        geometry=row['geometry']
    )

create_pois_batch(kg, poi, batch_size=10000)
create_roads_batch(kg, road, batch_size=10000)
create_blocks_from_poi(kg, poi, batch_size=5000)



  POI construction 1658/1658
  Road construction 5086/5086


100%|██████████| 1658/1658 [00:00<00:00, 39610.15it/s]


1028

In [39]:
print("=" * 80)
print("[1/8] Create POI-Area relationships")
print("=" * 80)
create_relationships_PLA_by_spatial(kg, poi, area, batch_size=5000)

print("\n" + "=" * 80)
print("[2/8] Create Area-Ward relationships")
print("=" * 80)
create_relationships_ABW_by_spatial(kg, area, ward, batch_size=5000)

print("\n" + "=" * 80)
print("[3/8] Create Road-Area relationships")
print("=" * 80)
create_relationships_RIA_by_spatial(kg, road, area, batch_size=10000)

print("\n" + "=" * 80)
print("[4/8] Create Area-Area border relationships")
print("=" * 80)
create_relationships_ABA_by_spatial(kg, area, batch_size=5000)

print("\n" + "=" * 80)
print("[5/8] Create POI-Block relationships")
print("=" * 80)
create_relationships_PLB_by_address(kg, poi, batch_size=5000)

print("\n" + "=" * 80)
print("[6/8] Create Block-Area relationships")
print("=" * 80)
create_relationships_BTA_by_address(kg, area, batch_size=5000)

print("\n" + "=" * 80)
print("[7/8] Create POI-Road nearby relationships")
print("=" * 80)
create_relationships_PNR_by_distance(kg, poi, road, max_distance=20, batch_size=5000)

print("\n" + "=" * 80)
print("[8/8] Enrich Block geometry")
print("=" * 80)
enrich_blocks_with_geometry(kg, block_data_path, batch_size=5000)

print("\n" + "=" * 80)
print("All KG relationships and enrichment steps completed")
print("=" * 80)



100%|██████████| 8240/8240 [00:00<00:00, 25147.70it/s]


In [35]:
print("=" * 80)
print("Create POI-Cate relationships")
print("=" * 80)
create_relationships_PCT_by_ptype(kg, poi, batch_size=10000)

Create POI-Cate relationships


100%|██████████| 1658/1658 [00:00<00:00, 34813.60it/s]

PCT created: 1982


In [48]:
kg.query("""
  MATCH (a:Area)-[:borderBy]->(b:Area)<-[:locatesAt]-(p:POI)-[:hasType]->(c:Cate)
  RETURN c.name, count(DISTINCT a) AS source_areas, count(DISTINCT b) AS neighbor_areas_with_cate
    """
)

[{'c.name': 'Factory', 'source_areas': 3140, 'neighbor_areas_with_cate': 3070},
 {'c.name': 'Business',
  'source_areas': 3140,
  'neighbor_areas_with_cate': 3022},
 {'c.name': 'Life_related',
  'source_areas': 3139,
  'neighbor_areas_with_cate': 3064},
 {'c.name': 'School', 'source_areas': 3098, 'neighbor_areas_with_cate': 2642},
 {'c.name': 'Car_service',
  'source_areas': 2711,
  'neighbor_areas_with_cate': 967},
 {'c.name': 'Medical_service',
  'source_areas': 3119,
  'neighbor_areas_with_cate': 2797},
 {'c.name': 'Meal', 'source_areas': 3121, 'neighbor_areas_with_cate': 2706},
 {'c.name': 'Shopping',
  'source_areas': 3133,
  'neighbor_areas_with_cate': 3002},
 {'c.name': 'Government',
  'source_areas': 3031,
  'neighbor_areas_with_cate': 1549},
 {'c.name': 'Amusement',
  'source_areas': 3123,
  'neighbor_areas_with_cate': 2551},
 {'c.name': 'others', 'source_areas': 3087, 'neighbor_areas_with_cate': 1973},
 {'c.name': 'Welfare', 'source_areas': 3059, 'neighbor_areas_with_cate': 2